In [3]:
#jay shree ram

**Fine-tuning Llama 3.2 3B(Learn with AI)**

In [ ]:
from huggingface_hub import login
HF_TOKEN = "your_huggingface_token_here"
login(token= HF_TOKEN)

In [5]:
!pip install -q -U trl transformers accelerate git+https://github.com/huggingface/peft.git
!pip install -q -U datasets bitsandbytes
!pip install trl

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.3 MB/s eta 0:00:00


In [6]:
import os
import torch
import transformers
from datasets import load_dataset
from trl import SFTTrainer
from peft import LoraConfig
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [7]:
model_id = 'meta-llama/Llama-3.2-3B-Instruct'
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_type = torch.bfloat16
)

In [8]:
lora_config = LoraConfig(
    r = 16,
    target_modules = ["q_proj", "v_proj"],
    task_type = "CAUSAL_LM"
)

In [9]:
tokenizer = AutoTokenizer.from_pretrained(model_id, token= HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(model_id,
                                             quantization_config= bnb_config,
                                             device_map= "auto",
                                             token= HF_TOKEN)

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [10]:
from datasets import load_dataset
dataset = load_dataset('openai/gsm8k', 'main')
# data = dataset.map(
#     lambda samples: tokenizer(samples['prompt'] + samples['text']),
#     batched=False
# )
data = dataset.map(
    lambda batch: tokenizer(
        [prompt + text for prompt, text in zip(batch['question'], batch['answer'])]
    ),
    batched=True,
    batch_size=32
)

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [11]:
def formatting_func(example):
  text = f"Question :{example ['prompt']}/nAnswer :{example ['text']}" #[0]}{tokenizer.eos_token}"
  return {"text":text}

In [12]:
!pip install -U transformers accelerate

In [15]:
trainer = SFTTrainer(
    train_dataset= data['train'],
    model= model,
    args = transformers.TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 2,
        warmup_steps = 200,
        max_steps = 3000,
        learning_rate = 3e-5,
        bf16 = True,
        fp16 = False,
        logging_steps = 1,
        output_dir = 'output',
        optim = 'paged_adamw_8bit',
        report_to = 'none'
        ),
    peft_config= lora_config,
    formatting_func= formatting_func
    )

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [16]:
trainer.train()
# trainer.save_model('output')

Step,Training Loss
1,1.426429
2,1.410053
3,1.104589
4,1.547869
5,1.600942
6,1.342428
7,1.297596
8,1.299994
9,1.493914
10,1.318243


TrainOutput(global_step=3000, training_loss=1.0673268316586813, metrics={'train_runtime': 8278.0261, 'train_samples_per_second': 0.725, 'train_steps_per_second': 0.362, 'total_flos': 1.6074646028132352e+16, 'train_loss': 1.0673268316586813})

In [18]:
trainer.model.save_pretrained("output")
tokenizer.save_pretrained("output")

('output/tokenizer_config.json',
 'output/chat_template.jinja',
 'output/tokenizer.json')

In [20]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [21]:
from transformers import pipeline
from peft import PeftModel

# Reload the base model in 4-bit
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map='auto',
    token=HF_TOKEN
)

# Load the PEFT adapter weights
FTmodel = PeftModel.from_pretrained(base_model, './output/')

# Merge the adapter weights with the base model
merged_model = FTmodel.merge_and_unload()

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [ ]:
from huggingface_hub import login
HF_TOKEN = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
login(token= HF_TOKEN)

In [24]:
# Change this to your desired repository ID
repo_id = "rajtembe13/Llama-3.2-3B-TUTOR-gsm8k"

# Push the merged model
merged_model.push_to_hub(repo_id)

# Push the tokenizer
tokenizer.push_to_hub(repo_id)

#model metadata
merged_model.config.use_cache = True
merged_model.config.push_to_hub(repo_id)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ne9owxf/model.safetensors:   0%|          | 22.5MB / 6.43GB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpxndsgsae/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/rajtembe13/Llama-3.2-3B-TUTOR-gsm8k/commit/e6d2ea96c01b75baa92779cdc7c3a11357a528a0', commit_message='Upload config', commit_description='', oid='e6d2ea96c01b75baa92779cdc7c3a11357a528a0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/rajtembe13/Llama-3.2-3B-TUTOR-gsm8k', endpoint='https://huggingface.co', repo_type='model', repo_id='rajtembe13/Llama-3.2-3B-TUTOR-gsm8k'), pr_revision=None, pr_num=None)

In [ ]:
# # Create a text generation pipeline
# pipeline = pipeline("text-generation", model=merged_model, tokenizer=tokenizer)

# # Test the fine-tuned model
# prompt = "question :what is a^2 * b^2."
# sequences = pipeline(
#     prompt,
#     max_new_tokens=50, # Generate up to 50 new tokens
#     do_sample=True,
#     top_k=10,
#     num_return_sequences=1,
#     eos_token_id=tokenizer.eos_token_id,
# )

# for seq in sequences:
#     print(f"Result: {seq['generated_text']}")